# Denoising Method Comparison — POC_DDM

Compares **three** denoising methods on raw kinetic (LAMP/DDM) curves, each applied with
its current chosen hyperparameter (no HP sweep re-run in this notebook):

| # | Method | Hyperparameter |
|---|---|---|
| 1 | Moving avg (`ori_curves_avg`) | `config.WINDOW_SIZE_ORI` |
| 2 | Wavelet (universal threshold) | `WAVELETS[0]` = `'sym8'`, `LEVEL`, `THRESH_MODE` |
| 3 | Savitzky-Golay | `SG_POLYORDER`, `SG_OPTIMAL_W` (fixed below) |


In [ ]:
import os, sys
import numpy as np
import matplotlib.pyplot as plt
import joblib
import pywt
from scipy.signal import savgol_filter
from scipy.ndimage import uniform_filter1d
from pathlib import Path

_NB_DIR = Path.cwd()
try:
    _nb = globals().get('__vsc_ipynb_file__')
    if _nb:
        _NB_DIR = Path(_nb).resolve().parent
except Exception:
    pass
_ROOT = _NB_DIR.parent.parent  # main_code/
sys.path.insert(0, str(_ROOT))
import config

try:
    from tqdm import tqdm
except ImportError:
    tqdm = None

print(f'pywt {pywt.__version__} | base: {config.BASE_FOLDER}')

In [ ]:
# ── Configuration ─────────────────────────────────────────────
# DATASET      = 'POC_DDM_final_nc_subtract'   # or 'POC_DDM_multi'
DATASET      = 'POC_DDM_final'
EXP_FOLDER   = os.path.join(config.BASE_FOLDER, DATASET)
CURVE_TYPE   = 'ori_curves'
WAVELETS     = ['sym8']          # first entry used in comparison
LEVEL        = None
THRESH_MODE  = 'soft'
SG_POLYORDER = 2                 # 2 = quadratic, 3 = cubic
SG_OPTIMAL_W = 69                # current chosen window -- derivative-test sweep on
                                  # D20260807_E00_C00_F4500KHz_U_DDM_02_07

In [ ]:
# ── Wavelet ────────────────────────────────────────────────────────────────
def denoise_curve(curve, wavelet, level=LEVEL, mode=THRESH_MODE):
    coeffs     = pywt.wavedec(curve, wavelet, level=level)
    sigma      = np.median(np.abs(coeffs[-1])) / 0.6745
    threshold  = sigma * np.sqrt(2 * np.log(len(curve)))
    new_coeffs = [coeffs[0]] + [pywt.threshold(d, threshold, mode=mode) for d in coeffs[1:]]
    return pywt.waverec(new_coeffs, wavelet)[:len(curve)]

def denoise_all(curves, wavelet):
    it = tqdm(curves, desc=f'Denoising [{wavelet}]', unit='curve') if tqdm else curves
    return np.array([denoise_curve(c, wavelet) for c in it])

# ── SG ─────────────────────────────────────────────────────────────────────────────
def _ensure_odd(w): return w + (1 - w % 2)

def apply_sg(curves, window_length, polyorder):
    w = _ensure_odd(int(window_length)); w = max(w, polyorder + 2)
    return savgol_filter(curves, window_length=w, polyorder=polyorder, axis=1)

# ── Derivative-test sweep -- drives the SG/Wavelet HP search below ─────────────────
def _derivative_scores(curves, param_values, denoise_fn, sample_size=400, seed=0):
    rng    = np.random.default_rng(seed)
    sample = curves[rng.choice(len(curves), size=min(sample_size, len(curves)), replace=False)]
    _ref_w = max(5, _ensure_odd(int(sample.shape[1] * 0.03)))
    ref    = savgol_filter(sample, window_length=_ref_w, polyorder=2, axis=1)
    peak_r = np.abs(np.diff(ref, axis=1)).max(axis=1)
    ro_raw = np.std(np.diff(np.diff(sample, axis=1), axis=1), axis=1)
    prs, ros = [], []
    for p in param_values:
        df = np.diff(denoise_fn(sample, p), axis=1)
        prs.append(np.mean(np.abs(df).max(axis=1) / np.where(peak_r > 0, peak_r, 1)))
        ros.append(np.mean(np.std(np.diff(df, axis=1), axis=1) / np.where(ro_raw > 0, ro_raw, 1)))
    return np.array(prs), np.array(ros)

def _sweet_spot(param_values, roughnesses):
    """First (smallest) param where roughness drops within 2x of its floor (10th pctile)."""
    floor = np.percentile(roughnesses, 10)
    sweet = np.where(roughnesses <= 2.0 * floor)[0]
    return float(param_values[sweet[0]] if len(sweet) else param_values[np.argmin(roughnesses)])

# ── Data loading ────────────────────────────────────────────────────────────────
def list_folders(exp_folder):
    out = []
    for name in sorted(os.listdir(exp_folder)):
        p = os.path.join(exp_folder, name)
        if (os.path.isdir(p) and name not in config.EXCLUDED_FOLDERS
                and os.path.exists(os.path.join(p, config.TRAINING_DATA_PATH))):
            out.append((name, p))
    return out

def load_exp(folder_path):
    d           = joblib.load(os.path.join(folder_path, config.TRAINING_DATA_PATH))
    curves      = np.array(d['curves'][CURVE_TYPE])
    if 'ori_curves_avg' in d['curves']:
        curves_avg = np.array(d['curves']['ori_curves_avg'])
    else:
        curves_avg = uniform_filter1d(np.array(d['curves']['ori_curves']), size=config.WINDOW_SIZE_ORI, axis=1, mode='nearest')
    well_labels = np.array(d['well_labels'])
    return curves, curves_avg, well_labels

print('All functions loaded.')

In [ ]:
import pandas as pd
from scipy.stats import pearsonr

# ── Colours & display constants ───────────────────────────────────────────────
METHOD_COLORS  = ['#CC79A7', '#E69F00', '#0072B2']
SG_POLY_COLORS = {2: '#0072B2', 3: '#009E73', 4: '#D55E00'}
_M_COLOR       = {'raw': '#999999', 'smoothed': '#CC79A7', 'sg': '#0072B2'}
_WAVELET_COLORS_CYCLE = ['#E69F00', '#56B4E9', '#009E73', '#F0E442', '#882255']

# HP search candidates (feed the SG/Wavelet HP search cells below)
SG_POLYORDERS      = [2, 3, 4]
WAVELET_CANDIDATES = [
    # 'db4', 'db6', 'db8',
    'sym4', 'sym6', 'sym8',
    # 'coif2', 'coif4',
    # 'bior3.5', 'bior4.4',
]

# ── Shared helpers ────────────────────────────────────────────────────────────
def _safe_corr(a, b):
    """Pearson r; returns np.nan if either input is constant."""
    return np.nan if np.std(a) == 0 or np.std(b) == 0 else pearsonr(a, b)[0]


def _full_metrics(raw, denoised):
    """Mean per-sample SNR, noise%, TV ratio, Pearson fidelity."""
    noise  = raw - denoised
    ns     = np.std(noise, axis=1)
    sr     = raw.max(axis=1) - raw.min(axis=1)
    vd, vn = np.var(denoised, axis=1), np.var(noise, axis=1)
    tv_d   = np.abs(np.diff(denoised, axis=1)).sum(axis=1)
    tv_r   = np.abs(np.diff(raw,      axis=1)).sum(axis=1)
    with np.errstate(divide='ignore', invalid='ignore'):
        snr_v   = np.where(vn > 0, 10*np.log10(np.where(vn > 0, vd/vn, 1.0)), np.nan)
        noise_v = np.where(sr > 0, ns / np.where(sr > 0, sr, 1.0) * 100, np.nan)
        tv_v    = tv_d / np.where(tv_r > 0, tv_r, np.nan)
    return {
        'snr':    float(np.nanmean(snr_v)),
        'noise%': float(np.nanmean(noise_v)),
        'tv':     float(np.nanmean(tv_v)),
        'corr':   float(np.nanmean([_safe_corr(raw[i], denoised[i]) for i in range(len(raw))])),
    }


# ── Method resolver ───────────────────────────────────────────────────────────
def _resolve_methods(r, methods):
    """Return [(array, title, color), ...] for the requested method keys.

    Keys:
      'smoothed'       moving average
      'sg'             SG (baseline polyorder + fixed window, see config)
      'sg_p2/3/4'      SG HP search result (per-polyorder auto window)
      'wv_<name>'      wavelet HP candidate  e.g. 'wv_sym6'
      '<name>'         baseline wavelet from WAVELETS  e.g. 'sym8'
    """
    wv_iter = iter(_WAVELET_COLORS_CYCLE)
    out = []
    for key in methods:
        if key == 'raw':
            out.append((r['raw'], 'Raw', _M_COLOR['raw']))
        elif key == 'smoothed':
            out.append((r['smoothed'],
                        f'Smoothed\n(w={config.WINDOW_SIZE_ORI})', _M_COLOR['smoothed']))
        elif key == 'sg':
            out.append((r.get('sg'),
                        f'SG p={SG_POLYORDER}\n(w={SG_OPTIMAL_W})', _M_COLOR['sg']))
        elif key.startswith('sg_p') and key[4:].isdigit():
            entry = r.get(key)
            if entry is not None:
                poly = int(key[4:])
                out.append((entry['denoised'],
                             f'SG p={poly}\n(w={entry["optimal_w"]})',
                             SG_POLY_COLORS.get(poly, '#888888')))
            else:
                print(f'[!] {key!r} not found — run SG HP search first')
        elif key.startswith('wv_'):
            wv_name = key[3:]
            entry   = r.get(key)
            if entry is not None:
                out.append((entry['denoised'],
                             f'Wavelet\n({wv_name})', next(wv_iter, '#888888')))
            elif wv_name in r.get('denoised', {}):
                out.append((r['denoised'][wv_name],
                             f'Wavelet\n({wv_name})', next(wv_iter, '#888888')))
            else:
                print(f'[!] {key!r} not found — run wavelet HP search first')
        elif key in r.get('denoised', {}):
            out.append((r['denoised'][key],
                        f'Wavelet\n({key})', next(wv_iter, '#888888')))
        else:
            print(f'[!] Unknown method key: {key!r}  (skipped)')
    return [(arr, lbl, col) for arr, lbl, col in out if arr is not None]


def _labels():
    return [
        f'Moving avg\n(w={config.WINDOW_SIZE_ORI})',
        f'Wavelet\n({WAVELETS[0]})',
        f'SG p={SG_POLYORDER}\n(w={SG_OPTIMAL_W})',
    ]

def _ready(r): return 'sg' in r


# ── Quantitative comparison ───────────────────────────────────────────────────
def compare_all_methods(folder_name, methods=None, plot=True):
    """
    Compute 7 denoising metrics and (optionally) plot bar charts.

    methods = None  → 3 baseline methods (smoothed, wavelet, sg), current hyperparams.
    methods = list  → any combination via _resolve_methods keys.

    Metrics: SNR, Noise%, Fidelity, AC lag-1, TV ratio, ΔTTP, SD_max ratio.
    """
    r = results[folder_name]
    if not _ready(r): print(f'[!] Run apply cell first for {folder_name}'); return
    raw = r['raw']

    if methods is None:
        labels  = _labels()
        arrays  = [r['smoothed'], r['denoised'][WAVELETS[0]], r['sg']]
        colors  = METHOD_COLORS
    else:
        resolved = _resolve_methods(r, methods)
        labels   = [l for _, l, _ in resolved]
        arrays   = [a for a, _, _ in resolved]
        colors   = [c for _, _, c in resolved]

    def tv(a): return np.abs(np.diff(a, axis=1)).sum(axis=1)
    def ac1(rw, dn):
        res = rw - dn
        return np.nanmean([np.corrcoef((e := res[i]-res[i].mean())[:-1], e[1:])[0, 1]
                           for i in range(len(res)) if res[i].std() > 0])

    mnames = ['SNR (dB)', 'Noise %', 'Fidelity\n(corr)', 'Residual\nAC lag-1',
              'TV ratio', 'Δ TTP', 'SD_max\nratio']
    better = ['↑', '↓', '↑', '↓', '↓', '↓', '→1']

    def best_idx(row, b):
        fin = np.isfinite(row)
        if not fin.any(): return None
        r_ = np.where(fin, row, np.nan)
        if b == '↓': return int(np.nanargmin(r_))
        if b == '↑': return int(np.nanargmax(r_))
        return int(np.nanargmin(np.abs(r_ - 1.0)))

    n_m  = len(labels)
    sc   = np.full((7, n_m), np.nan)
    _ref_w = max(5, _ensure_odd(int(raw.shape[1] * 0.03)))
    _ref   = savgol_filter(raw, window_length=_ref_w, polyorder=2, axis=1)
    draw   = np.abs(np.diff(_ref, axis=1)); ttp_raw = np.argmax(draw, axis=1).astype(float)
    for mi, den in enumerate(arrays):
        noise  = raw - den; ns = np.std(noise, axis=1); sr = raw.max(axis=1) - raw.min(axis=1)
        vd, vn = np.var(den, axis=1), np.var(noise, axis=1)
        dden   = np.abs(np.diff(den, axis=1))
        with np.errstate(divide='ignore', invalid='ignore'):
            sc[0,mi] = np.nanmean(np.where(vn > 0, 10*np.log10(np.where(vn > 0, vd/vn, 1.0)), np.nan))
            sc[1,mi] = np.nanmean(np.where(sr > 0, ns / np.where(sr > 0, sr, 1.0) * 100, np.nan))
        sc[2,mi] = np.nanmean([_safe_corr(raw[i], den[i]) for i in range(len(raw))])
        sc[3,mi] = ac1(raw, den)
        sc[4,mi] = np.nanmean(tv(den) / np.where(tv(raw) > 0, tv(raw), np.nan))
        sc[5,mi] = np.mean(np.abs(ttp_raw - np.argmax(dden, axis=1).astype(float)))
        sc[6,mi] = np.nanmean(dden.max(axis=1) / np.where(draw.max(axis=1)>0, draw.max(axis=1), np.nan))

    all_mn, all_sc, all_bt = mnames, list(sc), better

    if plot:
        fig, axes_pl = plt.subplots(1, len(all_mn), figsize=(max(6, 1.8*n_m), 5))
        if len(all_mn) == 1: axes_pl = [axes_pl]
        fig.suptitle(f'Quantitative Comparison — {folder_name}', fontsize=11, fontweight='bold')
        x = np.arange(n_m)
        for ax, metric, vals, b in zip(axes_pl, all_mn, all_sc, all_bt):
            best = best_idx(vals, b)
            for xi, (val, color) in enumerate(zip(vals, colors)):
                if np.isnan(val):
                    ax.bar(xi, 1, color='none', edgecolor=color, lw=1.5, ls='--', zorder=3)
                    ax.text(xi, 0.5, 'N/A', ha='center', va='center', fontsize=8, color=color, fontweight='bold')
                else:
                    bar = ax.bar(xi, val, color=color, edgecolor='black', zorder=3)
                    if xi == best: bar[0].set_edgecolor('red'); bar[0].set_linewidth(2.5)
                    ax.text(xi, val, f'{val:.3f}', ha='center', va='bottom', fontsize=7.5)
            ax.set_title(metric, fontweight='bold', fontsize=9)
            ax.set_xticks(x)
            ax.set_xticklabels([l.replace('\n',' ') for l in labels], rotation=30, ha='right', fontsize=8)
            ax.grid(axis='y', alpha=0.3, zorder=0)
        axes_pl[0].set_ylabel('↑ higher = better', fontsize=8, color='gray')
        if len(axes_pl) > 3: axes_pl[3].set_ylabel('↓ lower = better', fontsize=8, color='gray')
        if len(axes_pl) > 6: axes_pl[6].set_ylabel('→ 1.0 = best',     fontsize=8, color='gray')
        note = '■ red outline = best  |  N/A = zero noise variance'
        fig.text(0.99, 0.01, note, ha='right', fontsize=7.5, color='red', style='italic')
        fig.tight_layout(); plt.show(); plt.close(fig)

    # Build DataFrame (methods as rows, metrics as columns)
    directions = {mn.replace('\n', ' '): bt for mn, bt in zip(all_mn, all_bt)}
    df = pd.DataFrame(
        {mn.replace('\n', ' '): [float(v) for v in vals]
         for mn, vals in zip(all_mn, all_sc)},
        index=[l.replace('\n', ' ') for l in labels],
    )

    def _style_col(col):
        d      = directions.get(col.name, '↑')
        finite = col.dropna()
        if finite.empty:
            return [''] * len(col)
        best_lbl = ((finite - 1.0).abs().idxmin() if d == '→1'
                    else finite.idxmin()           if d == '↓'
                    else finite.idxmax())
        return ['background-color: #c8f7c5; font-weight: bold'
                if i == best_lbl else '' for i in col.index]

    try:
        from IPython.display import display
        display(df.style.apply(_style_col).format('{:.4f}', na_rep='N/A')
                  .set_caption(folder_name))
    except Exception:
        print(df.round(4).to_string())

    return df


print('All utility functions loaded.')

In [ ]:
label_maps = config.get_label_mappings(EXP_FOLDER)
results    = {}
folders    = list_folders(EXP_FOLDER)

print(f'{DATASET}: {len(folders)} folders')
for folder_name, folder_path in folders:
    print(f'\n─── {folder_name} ───')
    try:
        raw, smoothed, well_labels = load_exp(folder_path)
        print(f'  {raw.shape}')
        results[folder_name] = {
            'raw':         raw,
            'smoothed':    smoothed,
            'denoised':    {w: denoise_all(raw, wavelet=w) for w in WAVELETS},
            'well_labels': well_labels,
            'label_map':   label_maps.get(folder_name, {}),
        }
    except Exception as e:
        print(f'  [ERROR] {e}')

---
## Savitzky-Golay Smoothing

Fits a polynomial of degree $p$ to each sliding window of $w$ points. Unlike moving
average ($p=1$), it preserves peaks and the S-curve shape. `SG_OPTIMAL_W` above is the
current chosen window (fixed, not re-swept in this notebook).


---
## Hyperparameter Search (per chip)

SG (polyorder × window) and Wavelet (mother function) each use the same derivative-test
sweep (`_derivative_scores` / `_sweet_spot`) to auto-select their window/candidate per
chip — this is what feeds the `sg_p2/p3/p4` and `wv_sym4/sym6/sym8` rows in the table below.


In [ ]:
# ── SG HP Search: polyorder × window sweep ───────────────────────────────────
# Saves results[folder]['sg_p{poly}'] — re-running skips cached entries.

for folder_name, _ in folders:
    if folder_name not in results:
        continue
    r   = results[folder_name]
    raw = r['raw']
    T   = raw.shape[1]

    sg_windows = np.unique(np.array(
        [_ensure_odd(int(w)) for w in np.linspace(5, max(7, int(T * 0.25)), 40)]
    ))
    sg_windows = sg_windows[sg_windows >= 5]

    print(f'\n{folder_name}  (T={T})')
    for poly in SG_POLYORDERS:
        key = f'sg_p{poly}'
        if key in r:
            d = r[key]
            print(f'  SG p={poly}: cached  optimal_w={d["optimal_w"]}  '
                  f'SNR={d["metrics"]["snr"]:.1f}dB')
            continue
        fn       = lambda c, w, p=poly: apply_sg(c, w, p)
        prs, ros = _derivative_scores(raw, sg_windows.astype(float), fn)
        opt_w    = int(_sweet_spot(sg_windows.astype(float), ros))
        den      = apply_sg(raw, opt_w, poly)
        m        = _full_metrics(raw, den)
        r[key]   = dict(windows=sg_windows, prs=prs, ros=ros,
                        optimal_w=opt_w, denoised=den, metrics=m)
        print(f'  SG p={poly}: optimal_w={opt_w}  SNR={m["snr"]:.1f}dB  '
              f'TV={m["tv"]:.3f}  corr={m["corr"]:.4f}')

print('\nDone. Keys: sg_p2, sg_p3, sg_p4')

In [ ]:
# ── Wavelet HP Search: mother function comparison ─────────────────────────────
# Saves results[folder]['wv_<name>'] — re-running skips cached entries.

for folder_name, _ in folders:
    if folder_name not in results:
        continue
    r   = results[folder_name]
    raw = r['raw']
    T   = raw.shape[1]
    print(f'\n{folder_name}  (T={T})')

    for wv in WAVELET_CANDIDATES:
        key = f'wv_{wv}'
        if key in r:
            print(f'  Wavelet {wv}: cached')
            continue
        if wv in r.get('denoised', {}):
            den    = r['denoised'][wv]
            r[key] = dict(denoised=den, metrics=_full_metrics(raw, den), is_ref=True)
            print(f'  Wavelet {wv}: (already in WAVELETS)  SNR={r[key]["metrics"]["snr"]:.1f}dB')
            continue
        try:
            if pywt.Wavelet(wv).dec_len > T:
                print(f'  Wavelet {wv}: skip (filter > signal length)')
                continue
            den    = denoise_all(raw, wv)
            r[key] = dict(denoised=den, metrics=_full_metrics(raw, den), is_ref=False)
            m      = r[key]['metrics']
            print(f'  Wavelet {wv}: SNR={m["snr"]:.1f}dB  TV={m["tv"]:.3f}  '
                  f'corr={m["corr"]:.4f}')
        except Exception as e:
            print(f'  Wavelet {wv}: ERROR — {e}')

print('\nDone. Keys: wv_<name> for each candidate.')

In [ ]:
# ── Apply SG to all datasets ──────────────────────────────────────
# Override the current chosen window here if needed:
# SG_OPTIMAL_W = 21

print(f'SG    : w={SG_OPTIMAL_W}, p={SG_POLYORDER}')

for folder_name, _ in folders:
    if folder_name not in results: continue
    raw = results[folder_name]['raw']
    print(f'{folder_name}  ({raw.shape[0]} curves) ...', end=' ', flush=True)
    results[folder_name]['sg'] = apply_sg(raw, SG_OPTIMAL_W, SG_POLYORDER)
    print('done.')

print('\nAll methods applied.')

In [ ]:
denoising_metrics_by_folder = {}
for folder_name, folder_path in folders:
    denoising_metrics = compare_all_methods(
        folder_name, plot=False,
        methods=['smoothed', 'sg_p2', 'sg_p3', 'sg_p4', 'wv_sym4', 'wv_sym6', 'wv_sym8'])
    denoising_metrics_by_folder[folder_name] = denoising_metrics

### LaTeX table -- Residual AC lag-1 / SNR (dB) / Fidelity (corr) per chip

One sub-table per chip (`DDM_0x` -> `Chip 0x`), restricted to the three metrics
above. Methods are grouped as `Smoothed: Simple Moving Average` / `SG:
Savitzky-Golay` / `Wavelet: DWT`, with the per-row hyperparameter
(`w=`/`p=.. w=..`/`sym..`) split into its own column.

In [ ]:
import re
import string

def _chip_label(folder_name):
    m = re.search(r'DDM_(\d+)', folder_name)
    return f'Chip {int(m.group(1)):02d}' if m else folder_name


def _split_method_label(label):
    """'Smoothed (w=15)' -> ('Smoothed: Simple Moving Average', '(w=15)')
       'SG p=2 (w=31)'   -> ('SG: Savitzky-Golay', '(p=2 w=31)')
       'SG p=2 (w=~29)'  -> ('SG: Savitzky-Golay', '(p=2 w=~29)')  -- averaged-panel window
       'Wavelet (sym4)'  -> ('Wavelet: DWT', '(sym4)')"""
    m = re.match(r'^SG p=(\d+) \(w=(~?\d+)\)$', label)
    if m:
        return 'SG: Savitzky-Golay', f'(p={m.group(1)} w={m.group(2)})'
    m = re.match(r'^Smoothed \((w=\d+)\)$', label)
    if m:
        return 'Smoothed: Simple Moving Average', f'({m.group(1)})'
    m = re.match(r'^Wavelet \((.+)\)$', label)
    if m:
        return 'Wavelet: DWT', f'({m.group(1)})'
    return label, ''


_LATEX_METRIC_COLS      = ['Residual AC lag-1', 'SNR (dB)', 'Fidelity (corr)']
_LATEX_METRIC_HEADERS   = ['Residual Autocorrelation (Lag-1)', 'SNR (dB)', 'Fidelity (Corr)']
_LATEX_METRIC_FMT       = ['{:.3f}', '{:.2f}', '{:.3f}']
_LATEX_METRIC_DIRECTION = ['min', 'max', 'max']  # residual AC: lower is better; SNR/fidelity: higher is better


def _latex_panel(df, panel_letter, chip_name):
    best_row = {
        col: (df[col].idxmin() if direction == 'min' else df[col].idxmax())
        for col, direction in zip(_LATEX_METRIC_COLS, _LATEX_METRIC_DIRECTION)
    }

    lines = [
        f'    ({panel_letter}) Performance on {chip_name}\\\\[0.5em]',
        '    \\resizebox{\\textwidth}{!}{',
        '    \\begin{tabular}{llrrr}',
        '    \\toprule',
        '    \\textbf{Method} & \\textbf{Hyperparameter} & \\textbf{'
        + '} & \\textbf{'.join(_LATEX_METRIC_HEADERS) + '} \\\\',
        '    \\midrule',
    ]
    for method_label, row in df.iterrows():
        method_name, hp = _split_method_label(method_label)
        cells = []
        for col, fmt in zip(_LATEX_METRIC_COLS, _LATEX_METRIC_FMT):
            cell = fmt.format(row[col])
            if method_label == best_row[col]:
                cell = f'\\textbf{{{cell}}}'
            cells.append(cell)
        lines.append(f'    {method_name} & {hp} & {" & ".join(cells)} \\\\')
    lines += ['    \\bottomrule', '    \\end{tabular}', '    }']
    return '\n'.join(lines)


def _method_family_key(label):
    """Groups a method label the same way regardless of chip -- needed because SG HP
    search's per-chip auto-tuned window means the SAME method ('SG p=2') carries a
    DIFFERENT window in its label on every chip ('SG p=2 (w=31)' vs '(w=27)', ...).
    Averaging by the raw label string would treat those as different methods; this
    strips just the w=.. part so they group together. Every other label's
    hyperparameter (Smoothed's w, Wavelet's mother function) is a fixed constant
    across chips already, so it needs no stripping."""
    m = re.match(r'^(SG p=\d+) \(w=\d+\)$', label)
    return m.group(1) if m else label


def _extract_sg_window(label):
    m = re.match(r'^SG p=\d+ \(w=(\d+)\)$', label)
    return int(m.group(1)) if m else None


def _average_metrics_df(metrics_by_folder, folders):
    """Method x metric DataFrame averaged across every folder, grouped by
    _method_family_key (not the raw label) so SG HP search's per-chip window doesn't
    split what's really the same method into separate rows. The displayed SG window
    is the mean of each chip's own optimal_w, rounded to the nearest odd integer
    (matching _ensure_odd) and prefixed with '~' since chips didn't all land on the
    same value -- e.g. 'SG p=2 (w=~29)'."""
    used = [metrics_by_folder[f[0]] for f in folders if f[0] in metrics_by_folder]
    if not used:
        raise ValueError('No folders with metrics to average.')
    combined = pd.concat(used)
    family_keys = combined.index.map(_method_family_key)
    avg = combined.groupby(family_keys)[_LATEX_METRIC_COLS].mean()

    order, seen, sg_windows = [], set(), {}
    for lbl in used[0].index:
        fam = _method_family_key(lbl)
        if fam not in seen:
            seen.add(fam); order.append(fam)
    for df in used:
        for lbl in df.index:
            w = _extract_sg_window(lbl)
            if w is not None:
                sg_windows.setdefault(_method_family_key(lbl), []).append(w)

    def _display_label(fam):
        if fam in sg_windows:
            return f'{fam} (w=~{_ensure_odd(round(np.mean(sg_windows[fam])))})'
        return fam

    avg = avg.reindex(order)
    avg.index = [_display_label(f) for f in order]
    return avg


def build_denoising_latex_table(metrics_by_folder, folders, caption, label, include_average=True):
    used_folders = [f for f in folders if f[0] in metrics_by_folder]
    panels = [
        _latex_panel(metrics_by_folder[folder_name], string.ascii_lowercase[i], _chip_label(folder_name))
        for i, (folder_name, _) in enumerate(used_folders)
    ]
    if include_average and used_folders:
        avg_df = _average_metrics_df(metrics_by_folder, used_folders)
        panels.append(_latex_panel(avg_df, string.ascii_lowercase[len(used_folders)],
                                   f'Mean of All {len(used_folders)} Chips'))
    body = '\n\n    \\vspace{2.5em}\n\n'.join(panels)
    return (
        '\\begin{table}[htbp]\n'
        '    \\centering\n'
        f'    \\caption{{{caption}}}\n'
        f'    \\label{{{label}}}\n'
        '    \\small\n\n'
        f'{body}\n'
        '\\end{table}'
    )


denoising_latex = build_denoising_latex_table(
    denoising_metrics_by_folder, folders,
    caption='Denoising method comparison across the evaluated chips.',
    label='tab:denoising_comparison',
)
print(denoising_latex)

In [ ]:
mnames = ['Residual AC lag-1', 'SNR (dB)', 'Fidelity (corr)']
mascs = [True, False, False]
top_n = 5
for mname, masc in zip(mnames, mascs):
    print(f'\nTop {top_n} methods by {mname}:')
    display(denoising_metrics.sort_values(by=mname, ascending=masc)[[mname]].head(top_n))